In [76]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
전처리 과정 테스트 스크립트 - 일반 코드 버전
- FMP 데이터의 NaN 값 확인
- DB 보완 후 NaN 값 확인
- 각 단계별 데이터 상태 리포트
"""

import requests
import pandas as pd
import numpy as np
import calendar
import time

from tqdm import tqdm
import warnings
from sqlalchemy import create_engine
from DATA.stock_invest_function import *
from datetime import datetime
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')

# 설정값들
ticker = 'AMAT'

hs_code = '841191'

api_key = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

start_date = '2013-01-01'

# 유틸리티 함수들
def convert_to_month_end(date_str):
    try:
        # 문자열/타입 혼용 안전 변환
        date_obj = pd.to_datetime(date_str)
        if pd.isna(date_obj):
            return None

        y, m, d = date_obj.year, date_obj.month, date_obj.day

        # 1~5일 → 전달 말일
        if 1 <= d <= 5:
            if m == 1:
                prev_y, prev_m = y - 1, 12
            else:
                prev_y, prev_m = y, m - 1
            last_day_prev = calendar.monthrange(prev_y, prev_m)[1]
            return datetime(prev_y, prev_m, last_day_prev)

        # 그 외 → 해당월 말일
        last_day_cur = calendar.monthrange(y, m)[1]
        return datetime(y, m, last_day_cur)

    except Exception:
        return None

def create_monthly_end_dataframe(start_date):
    """
    start_date부터 이번달 전달까지 매월 말 기준으로
    날짜와 월 순서 더미 변수가 포함된 데이터프레임을 생성
    """
    start = datetime.strptime(start_date, '%Y-%m-%d')
    end = datetime.now().replace(day=1) - relativedelta(days=1)  # 이번달 전달 말일

    monthly_ends = []
    current_date = start.replace(day=1)

    while current_date <= end:
        next_month = current_date + relativedelta(months=1)
        month_end = next_month - relativedelta(days=1)

        if month_end <= end:
            monthly_ends.append(month_end)

        current_date = next_month

    return pd.DataFrame({
        'date_month_end': monthly_ends,
        'month_dummy': range(1, len(monthly_ends) + 1)
    })

def add_revenue_ttm(df):
    """Add TTM (Trailing Twelve Months) column to quarterly revenue data"""
    df_copy = df.copy()
    df_copy = df_copy.sort_values(['ticker', 'date'])
    ttm_values = []
    for ticker in df_copy['ticker'].unique():
        ticker_data = df_copy[df_copy['ticker'] == ticker].copy()
        ticker_data = ticker_data.sort_values('date')
        ticker_data['revenue_ttm'] = ticker_data['revenue'].rolling(window=4, min_periods=1).sum()
        ttm_values.extend(ticker_data['revenue_ttm'].tolist())
    df_copy['revenue_ttm'] = ttm_values
    return df_copy


def fetch_revenue_data(ticker, api_key):
    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {'limit': 200, 'apikey': api_key, 'period': 'quarter'}
    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code != 200:
            return None, f"HTTP {response.status_code}"
        data = response.json()
        if isinstance(data, dict) and 'Error Message' in data:
            return None, f"API 오류: {data['Error Message']}"
        if not data:
            return None, "데이터 없음"
        return data, None
    except Exception as e:
        return None, f"오류: {str(e)}"

def fetch_market_data_yearly(ticker, api_key, start_year=2010):
    all_data = []
    current_year = datetime.now().year
    for year in range(start_year, current_year + 1):
        start_date_str = f"{year}-01-01"
        end_date_str = f"{year}-12-31"
        url = f"https://financialmodelingprep.com/api/v3/historical-market-capitalization/{ticker}"
        params = {'from': start_date_str, 'to': end_date_str, 'apikey': api_key}
        try:
            response = requests.get(url, params=params, timeout=30)
            if response.status_code == 200:
                data = response.json()
                if data and isinstance(data, list):
                    all_data.extend(data)
            time.sleep(0.3)
        except Exception as e:
            continue
    return all_data if all_data else None, None

def process_daily_to_monthly_market_data(daily_data, ticker):
    if not daily_data:
        return pd.DataFrame()
    df = pd.DataFrame(daily_data)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df['year_month'] = df['date'].dt.to_period('M')
    monthly_data = []
    for year_month in df['year_month'].unique():
        month_data = df[df['year_month'] == year_month]
        last_day_data = month_data.loc[month_data['date'].idxmax()]
        monthly_data.append({
            'ticker': ticker,
            'date': last_day_data['date'],
            'market_cap': last_day_data['marketCap'],
            'market_cap_billions': round(last_day_data['marketCap'] / 1_000_000_000, 2),
        })
    return pd.DataFrame(monthly_data)

def fetch_db_revenue_data(ticker, db_info, end_date='2025-08-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, saleq
        FROM US_fundq
        WHERE ticker = '{ticker}'
        AND saleq IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['revenue_billions'] = df['saleq'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'revenue_billions']]
    except Exception as e:
        return pd.DataFrame()

def fetch_db_market_data(ticker, db_info, end_date='2024-12-31'):
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
            f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT date, ticker, me
        FROM US_fundm
        WHERE ticker = '{ticker}'
        AND me IS NOT NULL
        AND date <= '{end_date}'
        ORDER BY date ASC
        """
        df = pd.read_sql(query, con=engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
            df['market_cap_billions'] = df['me'] / 1000
            df['date_month_end'] = df['date'].apply(convert_to_month_end)
        return df[['ticker', 'date', 'date_month_end', 'market_cap_billions']]
    except Exception as e:
        return pd.DataFrame()

# ==============================================
# Export Data Collection Functions
# ==============================================

def get_hs_data(hs_code_6d, db_info):
    """Extract trade data by HS Code (2013-2024)"""
    try:
        engine = create_engine(
            f"mysql+pymysql://{db_info['user']}:{db_info['password']}@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        )
        query = f"""
        SELECT * FROM us_trade_monthly_data_with_forecast
        WHERE hs_code_6d = '{hs_code_6d}'
        AND date >= '2013-01-01'
        AND date <= '2026-12-31'
        ORDER BY date DESC
        """
        df = pd.read_sql(query, engine)
        engine.dispose()
        if not df.empty:
            df['date'] = pd.to_datetime(df['date'])
        return df
    except Exception:
        return pd.DataFrame()


def get_latest_input_date_data(df):
    """Extract data with the latest input_date only"""
    if 'input_date' not in df.columns:
        return pd.DataFrame()
    df_copy = df.copy()
    if not pd.api.types.is_datetime64_any_dtype(df_copy['input_date']):
        df_copy['input_date'] = pd.to_datetime(df_copy['input_date'])
    latest_date = df_copy['input_date'].max()
    latest_data = df_copy[df_copy['input_date'] == latest_date].copy()
    return latest_data


def collect_export_data(hs_code, db_info):
    """Collect export data (2013-2024)"""
    export_df = get_hs_data(hs_code, db_info)
    if export_df.empty:
        return pd.DataFrame()
    latest_export_data = get_latest_input_date_data(export_df)
    latest_export_data = latest_export_data.sort_values('date').reset_index(drop=True)
    if not latest_export_data.empty:
        latest_export_data['date'] = pd.to_datetime(latest_export_data['date'])
        latest_export_data['date_month_end'] = latest_export_data['date'].apply(convert_to_month_end)
    return latest_export_data



# ==============================================
# Data Merge and PSR Calculation Functions
# ==============================================

def calculate_enhanced_ttm_and_psr(merged_data):
    """Calculate enhanced TTM and PSR"""
    df = merged_data.copy()
    df = df.sort_values(['ticker', 'date_month_end']).reset_index(drop=True)

    # Calculate TTM from quarterly revenue
    df['revenue_ttm'] = df.groupby('ticker')['revenue_billions'].rolling(window=4, min_periods=1).sum().reset_index(0,
                                                                                                                    drop=True)
    df['revenue_ttm_billions'] = df['revenue_ttm']

    # Apply 2-month shift
    df['revenue_ttm_shift'] = df.groupby('ticker')['revenue_ttm_billions'].shift(2)

    # Calculate PSR
    df['PSR_ttm'] = df['market_cap_billions'] / df['revenue_ttm_shift']

    # Handle infinite values
    df['PSR_ttm'] = df['PSR_ttm'].replace([np.inf, -np.inf], np.nan)

    return df


def calculate_export_yoy_growth(df):
    """Calculate YoY growth rate for export data"""
    if 'expDlr' not in df.columns or df['expDlr'].isna().all():
        df['expDlr_yoy'] = pd.NA
        return df

    df = df.sort_values('date_month_end').reset_index(drop=True)
    df['expDlr_yoy'] = df['expDlr'].pct_change(periods=12) * 100  # YoY growth rate (%)

    return df


def merge_with_export_data(merged_data, export_data):
    """Merge final data with export data - preserve export forecasts"""
    if export_data.empty:
        merged_data['hs_code_6d'] = None
        merged_data['expDlr'] = pd.NA
        merged_data['expDlr_yoy'] = pd.NA
        return merged_data

    # Extract required columns from export data
    export_subset = export_data[['date_month_end', 'hs_code_6d', 'expDlr']].copy()

    # Calculate YoY growth rate
    export_subset = calculate_export_yoy_growth(export_subset)

    # Merge data (outer join to preserve all data)
    final_data = pd.merge(merged_data, export_subset, on='date_month_end', how='outer')

    # Sort by date
    final_data = final_data.sort_values('date_month_end').reset_index(drop=True)

    # Fill ticker NaN values with ffill
    if 'ticker' in final_data.columns:
        final_data['ticker'] = final_data['ticker'].ffill()
        # Apply bfill for cases where first row is NaN
        final_data['ticker'] = final_data['ticker'].bfill()

    return final_data

In [77]:
print("=" * 80)
print("전처리 과정 테스트 시작")
print(f"대상 종목: {ticker}")
print("=" * 80)

# 1. FMP 매출 데이터 수집
print("\n1. FMP 매출 데이터 수집 중...")
revenue_data, error = fetch_revenue_data(ticker, api_key)

if revenue_data is None:
    print(f"ERROR: FMP 매출 데이터 수집 실패 - {error}")
    exit()

all_revenue_data = []
for item in revenue_data:
    all_revenue_data.append({
        'ticker': ticker,
        'date': item.get('date', ''),
        'calendar_year': item.get('calendarYear', ''),
        'period': item.get('period', ''),
        'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
        'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
    })

fmp_revenue_df = pd.DataFrame(all_revenue_data)
fmp_revenue_df['date'] = pd.to_datetime(fmp_revenue_df['date'])
fmp_revenue_df = fmp_revenue_df.sort_values(['ticker', 'date'])
fmp_revenue_df['date_month_end'] = fmp_revenue_df['date'].apply(convert_to_month_end)
fmp_revenue_df = fmp_revenue_df.drop_duplicates(subset=['date_month_end'], keep='first').reset_index(drop=True)
print(f"FMP 매출 데이터: {len(fmp_revenue_df)}건")


# 2. DB 매출 데이터 가져오기
db_revenue_df = fetch_db_revenue_data(ticker, db_info)

# 3. FMP 시가총액 데이터 수집
print("2. FMP 시가총액 데이터 수집 중...")
market_data, error = fetch_market_data_yearly(ticker, api_key, start_year=2010)

if not market_data:
    print("ERROR: FMP 시가총액 데이터 수집 실패")
    exit()

fmp_market_df = process_daily_to_monthly_market_data(market_data, ticker)
fmp_market_df['date_month_end'] = fmp_market_df['date'].apply(convert_to_month_end)

print(f"FMP 시가총액 데이터: {len(fmp_market_df)}건")

# 4. FMP 원본 데이터의 NaN 값 확인
print("\n3. FMP 원본 데이터 NaN 분석")
print("-" * 50)

base_df = create_monthly_end_dataframe(start_date)

mereged_rev_data = pd.merge(base_df, fmp_revenue_df, on = 'date_month_end', how='outer')
mereged_rev_data.ffill(limit=2, inplace=True)
db_revenue_df.drop_duplicates(subset=['date_month_end'], inplace=True)

# 먼저 revenue 보충: date_month_end 기준 merge
mereged_rev_data = mereged_rev_data.merge(
    db_revenue_df[['date_month_end', 'revenue_billions']],
    on='date_month_end',
    how='left',
    suffixes=('', '_db')
)

# NaN 값 보충: revenue_billions가 NaN이면 db_revenue_df 값으로 채움
mereged_rev_data['revenue_billions'] = mereged_rev_data['revenue_billions'].fillna(
    mereged_rev_data['revenue_billions_db']
)

# 보조 컬럼 제거
# mereged_rev_data.drop(columns=['revenue_billions_db'], inplace=True)

mereged_rev_data[['ticker', 'calendar_year', 'period']] = mereged_rev_data[['ticker', 'calendar_year', 'period']].ffill()

merged_revenue_df = mereged_rev_data[['date_month_end', 'month_dummy', 'ticker', 'calendar_year', 'period', 'revenue_billions', 'revenue_billions_db']].copy()
# month_dummy가 NaN인 행 제거
merged_revenue_df= merged_revenue_df.dropna(subset=['month_dummy']).reset_index(drop=True)

전처리 과정 테스트 시작
대상 종목: AMAT

1. FMP 매출 데이터 수집 중...
FMP 매출 데이터: 159건
2. FMP 시가총액 데이터 수집 중...
FMP 시가총액 데이터: 189건

3. FMP 원본 데이터 NaN 분석
--------------------------------------------------


In [78]:
# 1. db_market_df 컬럼 이름 변경
db_market_df_renamed = db_market_df.rename(
    columns={'market_cap_billions': 'market_cap_billions_from_db'}
)

# 2. 두 데이터프레임 merge (date_month_end 기준)
merged_market_df = fmp_market_df.merge(
    db_market_df_renamed[['date_month_end', 'market_cap_billions_from_db']],
    on='date_month_end',
    how='outer'   # outer join으로 모든 데이터 보존
)

# 3. NaN 값 보충: market_cap_billions NaN이면 from_db 값으로 채움
merged_market_df['market_cap_billions'] = merged_market_df['market_cap_billions'].fillna(
    merged_market_df['market_cap_billions_from_db']
)

merged_market_df = merged_market_df.dropna(subset=['ticker'])
merged_market_df = merged_market_df.drop_duplicates(subset=['date_month_end'], keep='first')

enhanced_merged_df = pd.merge(merged_revenue_df, merged_market_df[['date_month_end', 'market_cap_billions']], on='date_month_end', how='inner')
enhanced_merged_df_with_ttm = calculate_enhanced_ttm_and_psr(enhanced_merged_df)

In [79]:
export_data = collect_export_data(hs_code, db_info)

In [80]:
finaal_df = merge_with_export_data(enhanced_merged_df_with_ttm, export_data)

In [81]:
finaal_df

,date_month_end,month_dummy,ticker,calendar_year,period,revenue_billions,revenue_billions_db,market_cap_billions,revenue_ttm,revenue_ttm_billions,revenue_ttm_shift,PSR_ttm,hs_code_6d,expDlr,expDlr_yoy
0,2013-01-31,1.0,AMAT,2013,Q1,1.57,1.573,15.14,1.57,1.57,NaN,NaN,841191,104240000.0,NaN
1,2013-02-28,2.0,AMAT,2013,Q1,1.57,1.573,16.08,3.14,3.14,NaN,NaN,841191,99986000.0,NaN
2,2013-03-31,3.0,AMAT,2013,Q1,1.57,1.573,15.81,4.71,4.71,1.57,10.070064,841191,97927000.0,NaN
3,2013-04-30,4.0,AMAT,2013,Q2,1.97,1.973,17.46,6.68,6.68,3.14,5.560510,841191,112503000.0,NaN
4,2013-05-31,5.0,AMAT,2013,Q2,1.97,1.973,18.29,7.08,7.08,4.71,3.883227,841191,82669500.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,2026-05-31,NaN,AMAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,841191,294712000.0,5.651990
161,2026-06-30,NaN,AMAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,841191,296792000.0,9.343846
162,2026-07-31,NaN,AMAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,841191,264542000.0,11.307838
163,2026-08-31,NaN,AMAT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,841191,301608000.0,9.782225


In [75]:
export_data

,hs_code_6d,date,expDlr,forecast,quarter,input_date,forecast_flag,created_at,date_month_end
0,841191,2013-01-31,104240000.0,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53,2013-01-31
1,841191,2013-02-28,99986000.0,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53,2013-02-28
2,841191,2013-03-31,97927000.0,0,2013Q1,2025-09-02,0,2025-09-02 17:14:53,2013-03-31
3,841191,2013-04-30,112503000.0,0,2013Q2,2025-09-02,0,2025-09-02 17:14:53,2013-04-30
4,841191,2013-05-31,82669500.0,0,2013Q2,2025-09-02,0,2025-09-02 17:14:53,2013-05-31
...,...,...,...,...,...,...,...,...,...
139,841191,2024-08-31,253650000.0,0,2024Q3,2025-09-02,0,2025-09-02 17:14:53,2024-08-31
140,841191,2024-09-30,212766000.0,0,2024Q3,2025-09-02,0,2025-09-02 17:14:53,2024-09-30
141,841191,2024-10-31,247061000.0,0,2024Q4,2025-09-02,0,2025-09-02 17:14:53,2024-10-31
142,841191,2024-11-30,255786000.0,0,2024Q4,2025-09-02,0,2025-09-02 17:14:53,2024-11-30
